Step 1 — Install Dependencies

In [ ]:
!pip install yfinance scikit-learn ta xgboost lightgbm catboost gradio --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00


Step 2 — Imports & Config

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import datetime, io
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from PIL import Image
import gradio as gr
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from ta.momentum   import RSIIndicator, StochasticOscillator, WilliamsRIndicator, ROCIndicator
from ta.trend      import MACD, EMAIndicator, SMAIndicator, CCIIndicator, ADXIndicator
from ta.volatility import BollingerBands, AverageTrueRange, KeltnerChannel
from ta.volume     import OnBalanceVolumeIndicator, MFIIndicator

TICKER, INTERVAL, YEARS, THRESHOLD, N_SPLITS = "TCS.NS", "1h", 2, 0.55, 5
MODEL_XGB = MODEL_LGB = MODEL_CAT = SCALER = FEAT_COLS = None
MODEL_ACC = MODEL_AUC = 0.0
IS_TRAINED = False

UP,DOWN,GOLD,BG,PBG,TXT,BLUE,PINK,GRID = ('#00ff9f','#ff4757','#ffd700','#0b0f1a', '#0f1623','#94a3b8','#38bdf8','#f472b6','#1a2235')
print(f"   Ticker   : {TICKER}")
print(f"   Interval : {INTERVAL}")
print(f"   History  : {YEARS} years")

   Ticker   : TCS.NS
   Interval : 1h
   History  : 2 years


Step 3 — Data Fetching

In [ ]:
def clean(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[[c for c in ['Open','High','Low','Close','Volume'] if c in df.columns]].copy()
    return df[df['Volume'] > 0].dropna()

def fetch_hist():
    frames, end = [], datetime.datetime.today()
    for i in range((YEARS * 365) // 55 + 1):
        e = end - datetime.timedelta(days=i*55)
        s = e   - datetime.timedelta(days=55)
        try:
            df = yf.download(TICKER, start=s.strftime('%Y-%m-%d'), end=e.strftime('%Y-%m-%d'), interval=INTERVAL, progress=False, auto_adjust=True)
            if df is not None and not df.empty:
                frames.append(clean(df))
        except: pass
    return pd.concat(frames).drop_duplicates().sort_index()

def fetch_live():
    for p in ['7d','14d','30d']:
        df = yf.download(TICKER, period=p, interval=INTERVAL, progress=False, auto_adjust=True)
        df = clean(df) if df is not None and not df.empty else pd.DataFrame()
        if len(df) >= 60: return df
    return df

Step 4 — Feature Engineering

In [ ]:
def sq(s):
  return s.iloc[:,0].squeeze() if isinstance(s, pd.DataFrame) else s.squeeze()

def features(df):
  df = df.copy()
  C,H,L,V,O = sq(df['Close']),sq(df['High']),sq(df['Low']),sq(df['Volume']),sq(df['Open'])

  df['EMA_5']  = EMAIndicator(C,5).ema_indicator()
  df['EMA_9']  = EMAIndicator(C,9).ema_indicator()
  df['EMA_21'] = EMAIndicator(C,21).ema_indicator()
  df['EMA_50'] = EMAIndicator(C,50).ema_indicator()
  df['SMA_20'] = SMAIndicator(C,20).sma_indicator()
  df['SMA_50'] = SMAIndicator(C,50).sma_indicator()
  m = MACD(C)
  df['MACD'],df['MACD_Signal'],df['MACD_Hist'] = m.macd(),m.macd_signal(),m.macd_diff()
  df['ADX'] = ADXIndicator(H,L,C,14).adx()
  df['CCI'] = CCIIndicator(H,L,C,20).cci()
  df['EMA5_9_cross']   = (df['EMA_5']  - df['EMA_9'])  / C
  df['EMA9_21_cross']  = (df['EMA_9']  - df['EMA_21']) / C
  df['EMA21_50_cross'] = (df['EMA_21'] - df['EMA_50']) / C
  df['Price_EMA21']    = (C - df['EMA_21']) / df['EMA_21']
  df['Price_SMA50']    = (C - df['SMA_50']) / df['SMA_50']

  df['RSI_14']    = RSIIndicator(C,14).rsi()
  df['RSI_6']     = RSIIndicator(C,6).rsi()
  df['RSI_delta'] = df['RSI_14'] - df['RSI_14'].shift(1)
  st = StochasticOscillator(H,L,C,14)
  df['Stoch_K'],df['Stoch_D'] = st.stoch(),st.stoch_signal()
  df['Stoch_KD_diff'] = df['Stoch_K'] - df['Stoch_D']
  df['WilliamsR']     = WilliamsRIndicator(H,L,C,14).williams_r()
  df['ROC_5'],df['ROC_10'] = ROCIndicator(C,5).roc(),ROCIndicator(C,10).roc()

  bb = BollingerBands(C,20,2)
  df['BB_Upper'],df['BB_Lower'] = bb.bollinger_hband(),bb.bollinger_lband()
  df['BB_Width'],df['BB_Pct']   = bb.bollinger_wband(),bb.bollinger_pband()
  kc = KeltnerChannel(H,L,C,20)
  df['KC_Upper'],df['KC_Lower'] = kc.keltner_channel_hband(),kc.keltner_channel_lband()
  df['ATR']     = AverageTrueRange(H,L,C,14).average_true_range()
  df['ATR_pct'] = df['ATR'] / C

  df['OBV']          = OnBalanceVolumeIndicator(C,V).on_balance_volume()
  df['MFI']          = MFIIndicator(H,L,C,V,14).money_flow_index()
  df['Vol_MA_10']    = V.rolling(10).mean()
  df['Vol_Ratio']    = V / (df['Vol_MA_10'] + 1e-9)
  df['Vol_Change']   = V.pct_change()
  df['Vol_MA_ratio'] = V.rolling(5).mean() / (V.rolling(20).mean() + 1e-9)

  df['Return_1'],df['Return_3']  = C.pct_change(1),C.pct_change(3)
  df['Return_5'],df['Return_10'] = C.pct_change(5),C.pct_change(10)
  df['HL_pct']       = (H - L) / (C + 1e-9)
  df['CO_pct']       = (C - O) / (O + 1e-9)
  df['Gap_pct']      = (O - C.shift(1)) / (C.shift(1) + 1e-9)
  df['Upper_shadow'] = (H - np.maximum(C,O)) / (H - L + 1e-9)
  df['Lower_shadow'] = (np.minimum(C,O) - L) / (H - L + 1e-9)
  df['Body_ratio']   = abs(C - O) / (H - L + 1e-9)

  for lag in [1,2,3,5,8,13]:
    df[f'Lag_ret_{lag}'] = C.pct_change().shift(lag)

  df['Vol_10d']       = C.pct_change().rolling(10).std()
  df['Vol_20d']       = C.pct_change().rolling(20).std()
  df['Momentum_5']    = C - C.shift(5)
  df['Momentum_10']   = C - C.shift(10)
  df['Close_rank_20'] = C.rolling(20).rank(pct=True)
  df['Close_rank_50'] = C.rolling(50).rank(pct=True)

  df['Target'] = (C.shift(-1) > C).astype(int)
  df.replace([np.inf,-np.inf], np.nan, inplace=True)
  df.dropna(inplace=True)
  return df

Step 5 — Train

In [ ]:
EXCL = ['Target','Open','High','Low','Close','Volume', 'BB_Upper','BB_Lower','KC_Upper','KC_Lower', 'EMA_5','EMA_9','EMA_21','EMA_50','SMA_20','SMA_50']

def make_models():
  return (
      XGBClassifier(n_estimators=600, max_depth=5, learning_rate=0.02, subsample=0.8, colsample_bytree=0.75, min_child_weight=5,
                    gamma=0.15, reg_alpha=0.1, reg_lambda=1.0, eval_metric='logloss', use_label_encoder=False, random_state=42, n_jobs=-1),
      LGBMClassifier(n_estimators=600, max_depth=5, learning_rate=0.02, subsample=0.8, colsample_bytree=0.75, min_child_samples=20,
                     random_state=42, n_jobs=-1, verbose=-1),
      CatBoostClassifier(iterations=400, depth=5, learning_rate=0.03, l2_leaf_reg=3, random_seed=42, verbose=0)
  )

def train(df):
  fc = [c for c in df.columns if c not in EXCL]
  X, y = df[fc].values, df['Target'].values
  sc = StandardScaler()
  accs, aucs = [], []

  for tr, te in TimeSeriesSplit(n_splits=N_SPLITS).split(X):
    Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
    ytr, yte = y[tr], y[te]
    xgb, lgb, cat = make_models()
    xgb.fit(Xtr, ytr, verbose=False)
    lgb.fit(Xtr, ytr)
    cat.fit(Xtr, ytr)
    p = (xgb.predict_proba(Xte)[:,1] + lgb.predict_proba(Xte)[:,1] + cat.predict_proba(Xte)[:,1]) / 3
    accs.append(accuracy_score(yte, (p >= THRESHOLD).astype(int)))
    aucs.append(roc_auc_score(yte, p))

  sc_f = StandardScaler()
  Xa   = sc_f.fit_transform(X)
  xgb_f, lgb_f, cat_f = make_models()
  xgb_f.fit(Xa, y, verbose=False)
  lgb_f.fit(Xa, y)
  cat_f.fit(Xa, y)
  return xgb_f, lgb_f, cat_f, sc_f, fc, np.mean(accs), np.mean(aucs)

Step 6 — Predict

In [ ]:
def predict(df_raw):
    df = features(df_raw)
    for c in FEAT_COLS:
        if c not in df.columns: df[c] = 0.0
    X = SCALER.transform(df[FEAT_COLS].values)
    p = (MODEL_XGB.predict_proba(X)[:,1] +
         MODEL_LGB.predict_proba(X)[:,1] +
         MODEL_CAT.predict_proba(X)[:,1]) / 3
    df['Prob_UP']    = p
    df['Prob_DOWN']  = 1 - p
    df['Prediction'] = (p >= THRESHOLD).astype(int)
    df['Signal']     = ['BUY' if x else 'SELL' for x in df['Prediction']]
    return df

Step 7 — Plots

In [ ]:
def fig2pil(fig):
  buf = io.BytesIO()
  fig.savefig(buf, format='png', dpi=130, bbox_inches='tight', facecolor=fig.get_facecolor())
  buf.seek(0); img = Image.open(buf).copy()
  buf.close(); plt.close(fig)
  return img

def plot_lhs(df_pred):
  r = df_pred.tail(80).copy(); n = len(r); x = np.arange(n)
  fig = plt.figure(figsize=(14,9), facecolor=BG)
  gs  = gridspec.GridSpec(3,1, figure=fig, hspace=0.04, top=0.95, bottom=0.07, left=0.07, right=0.97, height_ratios=[4,1.2,1.2])
  close = r['Close'].values.flatten(); open_ = r['Open'].values.flatten()
  high_ = r['High'].values.flatten();  low_  = r['Low'].values.flatten()

  ax1 = fig.add_subplot(gs[0]); ax1.set_facecolor(PBG)
  colors = [UP if close[i]>=open_[i] else DOWN for i in range(n)]
  ax1.bar(x, close-open_, bottom=open_, color=colors, width=0.7, alpha=0.9, zorder=3)
  ax1.vlines(x, low_, high_, color=colors, lw=0.6, zorder=2)
  ax1.plot(x, r['EMA_9'].values,  color=BLUE, lw=1.2, label='EMA-9',  zorder=4, alpha=0.85)
  ax1.plot(x, r['EMA_21'].values, color=PINK, lw=1.2, label='EMA-21', zorder=4, alpha=0.85)
  ax1.fill_between(x, r['BB_Upper'].values, r['BB_Lower'].values, alpha=0.05, color='#818cf8')

  for i,(_, row) in enumerate(r.iterrows()):
    if   row.get('Signal') == 'BUY':
      ax1.scatter(i, float(row['Low'])*0.9985,  marker='^', color=UP,   s=30, zorder=6)
    elif row.get('Signal') == 'SELL':
      ax1.scatter(i, float(row['High'])*1.0015, marker='v', color=DOWN, s=30, zorder=6)
  ax1.set_title('  TCS.NS  ·  1H', color=TXT, fontsize=10, pad=5, loc='left', fontfamily='monospace')
  ax1.legend(fontsize=7.5, facecolor=PBG, labelcolor=TXT, framealpha=0.4, loc='upper left', ncol=2)
  ax1.tick_params(colors=TXT, labelsize=7, bottom=False, labelbottom=False)
  ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('₹%.0f'))
  for sp in ax1.spines.values(): sp.set_color(GRID)

  ax2 = fig.add_subplot(gs[1]); ax2.set_facecolor(PBG)
  rv = r['RSI_14'].values
  ax2.plot(x, rv, color=GOLD, lw=1.3, zorder=3)
  ax2.axhline(70, color=DOWN, ls='--', lw=0.7, alpha=0.6)
  ax2.axhline(30, color=UP,   ls='--', lw=0.7, alpha=0.6)
  ax2.fill_between(x, rv, 70, where=(rv>70), alpha=0.10, color=DOWN)
  ax2.fill_between(x, rv, 30, where=(rv<30), alpha=0.10, color=UP)
  ax2.set_ylim(5,95); ax2.set_yticks([30,70])
  ax2.set_ylabel('RSI', color=TXT, fontsize=8, fontfamily='monospace')
  ax2.tick_params(colors=TXT, labelsize=7, bottom=False, labelbottom=False)
  for sp in ax2.spines.values(): sp.set_color(GRID)

  ax3 = fig.add_subplot(gs[2]); ax3.set_facecolor(PBG)
  hv = r['MACD_Hist'].values
  ax3.bar(x, hv, color=[UP if v>=0 else DOWN for v in hv], alpha=0.75, width=0.7, zorder=3)
  ax3.plot(x, r['MACD'].values,        color=BLUE, lw=1.0)
  ax3.plot(x, r['MACD_Signal'].values, color=PINK, lw=1.0)
  ax3.axhline(0, color=GRID, lw=0.5)
  ax3.set_ylabel('MACD', color=TXT, fontsize=8, fontfamily='monospace')
  ax3.tick_params(colors=TXT, labelsize=7)
  for sp in ax3.spines.values(): sp.set_color(GRID)
  step = max(1, n//8); tpos = list(range(0,n,step))
  ax3.set_xticks(tpos)
  ax3.set_xticklabels([r.index[i].strftime('%d %b %H:%M') for i in tpos],  rotation=25, ha='right', fontsize=6.5, color='#475569')
  fig.patch.set_facecolor(BG)
  return fig

def plot_rhs(df_raw):
  r   = df_raw.tail(60).copy()
  cv  = r['Close'].values.flatten(); xv = np.arange(len(cv))
  lp  = float(cv[-1]); pp = float(cv[-2]) if len(cv)>=2 else lp
  chg = lp - pp; pct = chg/pp*100; col = UP if chg>=0 else DOWN
  ts  = r.index[-1]
  ts_str = ts.strftime('%d %b %Y  %H:%M') if hasattr(ts,'strftime') else str(ts)
  dh  = float(r['High'].values.flatten()[-20:].max())
  dl  = float(r['Low'].values.flatten()[-20:].min())
  vol = float(df_raw['Volume'].values.flatten()[-1])

  fig = plt.figure(figsize=(5.5,9), facecolor=BG)
  gs  = gridspec.GridSpec(2,1, figure=fig, hspace=0.08, top=0.93, bottom=0.03, left=0.04, right=0.97, height_ratios=[1.2,2])
  ax1 = fig.add_subplot(gs[0]); ax1.set_facecolor(PBG)
  ax1.plot(xv, cv, color=col, lw=2.0, zorder=3)
  ax1.fill_between(xv, cv, cv.min()*0.9995, alpha=0.12, color=col)
  ax1.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
  for sp in ax1.spines.values(): sp.set_color(GRID)
  ax1.set_title('  TCS.NS  ·  LIVE', color=TXT, fontsize=9, pad=4, loc='left', fontfamily='monospace')
  ax2 = fig.add_subplot(gs[1]); ax2.set_facecolor(PBG)
  ax2.axis('off'); ax2.set_xlim(0,1); ax2.set_ylim(0,1)

  def t(x, y, s, c=TXT, fs=10, fw='normal', ha='center'):
    ax2.text(x, y, s, color=c, fontsize=fs, fontweight=fw, ha=ha, va='center', fontfamily='monospace', transform=ax2.transAxes)

  arr = 'UP' if chg>=0 else 'DOWN'
  t(0.5, 0.85, f'₹{lp:,.2f}',                       c=col,      fs=28, fw='bold')
  t(0.5, 0.70, f'{arr}  {chg:+.2f}  ({pct:+.2f}%)', c=col,      fs=12)
  ax2.plot([0.1,0.9],[0.62,0.62], color=GRID, lw=0.8, transform=ax2.transAxes)
  t(0.5,  0.54, ts_str,            c='#475569', fs=10)
  t(0.25, 0.42, 'DAY HIGH',        c='#334155', fs=8)
  t(0.75, 0.42, 'DAY LOW',         c='#334155', fs=8)
  t(0.25, 0.34, f'₹{dh:,.2f}',    c=UP,        fs=11, fw='bold')
  t(0.75, 0.34, f'₹{dl:,.2f}',    c=DOWN,      fs=11, fw='bold')
  t(0.5,  0.22, 'VOLUME',          c='#334155', fs=8)
  t(0.5,  0.14, f'{vol/1000:.0f}K',c='#475569', fs=13, fw='bold')
  fig.patch.set_facecolor(BG)
  return fig

Step 8 — Signal HTML

In [ ]:
def make_signal_html(df_pred):
  last    = df_pred.iloc[-1]
  prob_up = float(last['Prob_UP']); prob_dn = float(last['Prob_DOWN'])
  close   = float(last['Close']);   is_buy  = int(last['Prediction']) == 1
  ts_str  = df_pred.index[-1].strftime('%d %b %Y  %H:%M IST')
  sig_txt = "BUY"       if is_buy else "SELL"
  sig_icon= "UP"         if is_buy else "DOWN"
  sig_clr = "#00ff9f"   if is_buy else "#ff4757"
  sub_txt = "PRICE GOING UP"   if is_buy else "PRICE GOING DOWN"
  act_txt = "ENTER LONG"       if is_buy else "EXIT / SHORT"
  bw = int(prob_up*100); dw = 100 - bw
  return f"""
  <div style="background:#0b0f1a;border:1.5px solid {sig_clr}33;border-radius:16px; padding:32px 20px;font-family:'JetBrains Mono',monospace;color:#e2e8f0;
  display:flex;flex-direction:column;align-items:center; min-height:560px;justify-content:center;">
    <div style="width:100%;background:#0f1623;border:2px solid {sig_clr}; border-radius:14px;padding:32px 16px;text-align:center;
    margin-bottom:24px;box-shadow:0 0 40px {sig_clr}12;">
    <div style="font-size:80px;font-weight:900;color:{sig_clr};line-height:1;">
    {sig_icon} {sig_txt}</div>
    <div style="font-size:13px;color:{sig_clr};opacity:0.8; margin-top:8px;letter-spacing:2px;">{sub_txt}</div>
    <div style="font-size:12px;color:#475569;margin-top:6px;">→ {act_txt}</div>
  </div>
  <div style="width:100%;background:#0f1623;border:1px solid #1a2235; border-radius:10px;padding:14px;text-align:center;margin-bottom:20px;">
    <div style="font-size:11px;color:#334155;letter-spacing:2px;margin-bottom:4px;">PRICE</div>
    <div style="font-size:28px;font-weight:900;color:#f1f5f9;">₹{close:,.2f}</div>
  </div>
  <div style="width:100%;margin-bottom:6px;">
    <div style="display:flex;justify-content:space-between;margin-bottom:5px;">
      <span style="font-size:11px;color:#00ff9f;font-weight:700;">UP {prob_up:.0%}</span>
      <span style="font-size:11px;color:#ff4757;font-weight:700;">DOWN {prob_dn:.0%}</span>
    </div>
    <div style="height:10px;border-radius:6px;overflow:hidden;display:flex;width:100%;">
    <div style="width:{bw}%;background:#00ff9f;"></div>
    <div style="width:{dw}%;background:#ff4757;"></div>
    </div>
  </div>
  <div style="font-size:11px;color:#334155;margin-top:14px; text-align:center;letter-spacing:1px;">{ts_str}</div>
  </div>"""


def loading_html():
  return """<div style="background:#0b0f1a;border:1px solid #1a2235;border-radius:16px; padding:60px 20px;font-family:monospace;text-align:center;color:#334155;
  min-height:560px;display:flex;flex-direction:column;justify-content:center;align-items:center;gap:14px;">
  <div style="font-size:28px;color:#1e293b;">SETTINGS</div>
  <div style="font-size:12px;color:#334155;line-height:2;">
  Loading data…<br>Training model…<br>Fetching live feed…</div>
  </div>"""

Step 9 — Dashboard Logic

In [ ]:
def build_dashboard():
  global MODEL_XGB, MODEL_LGB, MODEL_CAT, SCALER, FEAT_COLS
  global MODEL_ACC, MODEL_AUC, IS_TRAINED
  if not IS_TRAINED:
    df_feat = features(fetch_hist())
    (MODEL_XGB, MODEL_LGB, MODEL_CAT, SCALER, FEAT_COLS, MODEL_ACC, MODEL_AUC) = train(df_feat)
    IS_TRAINED = True
  df_raw  = fetch_live()
  df_pred = predict(df_raw)
  return fig2pil(plot_lhs(df_pred)), make_signal_html(df_pred), fig2pil(plot_rhs(df_raw))

def refresh_dashboard():
  if not IS_TRAINED: return None, loading_html(), None
  df_raw  = fetch_live()
  df_pred = predict(df_raw)
  return fig2pil(plot_lhs(df_pred)), make_signal_html(df_pred), fig2pil(plot_rhs(df_raw))

Step 10 — Gradio UI

In [ ]:
CSS = """
body,.gradio-container{background:#0b0f1a !important;}
button{background:#0f1623 !important;border:1px solid #1a2235 !important; color:#94a3b8 !important;border-radius:8px !important;font-family:monospace !important;}
button:hover{border-color:#00ff9f !important;color:#00ff9f !important;}
img{border-radius:10px !important;} footer{display:none !important;} """

HEADER = """
<div style="padding:14px 20px 10px;border-bottom:1px solid #1a2235; margin-bottom:10px;font-family:monospace;">
<span style="font-size:16px;font-weight:900; background:linear-gradient(90deg,#00ff9f,#38bdf8);-webkit-background-clip:text;-webkit-text-fill-color:transparent;">
TCS.NS    REAL-TIME ML SIGNAL
</span>
<span style="font-size:9px;color:#1e293b;margin-left:16px;letter-spacing:1px;">
1H    LIVE
</span>
</div>"""

with gr.Blocks(css=CSS, title="TCS.NS Signal") as app:
  gr.HTML(HEADER)
  with gr.Row():
    gr.HTML('<div style="flex:1;"></div>')
    btn = gr.Button("REFRESH", scale=0, min_width=140)
  with gr.Row(equal_height=True):
    with gr.Column(scale=5):
      lhs_out = gr.Image(label="", show_label=False, height=580, container=False, show_download_button=False)
    with gr.Column(scale=3):
      signal_out = gr.HTML(value=loading_html())
    with gr.Column(scale=3):
      rhs_out = gr.Image(label="", show_label=False, height=580, container=False, show_download_button=False)
  btn.click(fn=refresh_dashboard, outputs=[lhs_out, signal_out, rhs_out])
  app.load(fn=build_dashboard,    outputs=[lhs_out, signal_out, rhs_out])
app.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://27c388db52cdfe1e27.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
